# Proyecto Final – Python for ETL

Proceso ETL que lee `server_inputs/file.ope`, genera los DataFrames **cliente** y **deuda**, y los guarda en `server_outputs` (`cliente.csv`, `deuda.csv` y `deuda.db` en SQLite).

Este notebook contiene **el mismo código que `main.py`**, dividido en celdas: **EXTRACT**, **TRANSFORM** y **LOAD**.

**Cómo usarlo en Google Colab:** menú **Entorno de ejecución → Ejecutar todas** (o ejecutar las celdas en orden con Shift + Enter). El Paso 0 pedirá subir el archivo `file.ope`. La guía completa está en `GUIA_COLAB.md`.

## Paso 0 – Preparar las carpetas (Google Colab)

Crea `server_inputs` y `server_outputs`. Si `file.ope` todavía no está en `server_inputs`, abre una ventana para subirlo desde tu computadora.

In [ ]:
from pathlib import Path

Path("server_inputs").mkdir(exist_ok=True)
Path("server_outputs").mkdir(exist_ok=True)

if Path("server_inputs/file.ope").exists():
    print("file.ope ya está en la carpeta server_inputs")
else:
    from google.colab import files
    print("Selecciona el archivo file.ope desde tu computadora:")
    archivos_subidos = files.upload()
    nombre_subido = list(archivos_subidos.keys())[0]
    Path(nombre_subido).rename("server_inputs/file.ope")
    print(f"'{nombre_subido}' se guardó como server_inputs/file.ope")

## Librerías, rutas y columnas

In [1]:
from pathlib import Path
import sqlite3

import pandas as pd


# ============================================================================
# CONFIGURACIÓN: rutas y nombres de columnas
# ============================================================================

# Carpeta donde está este archivo main.py. Todas las rutas se construyen a
# partir de ella, así el proyecto funciona en cualquier computadora.
# En un notebook (Google Colab / Jupyter) no existe __file__, por eso en ese
# caso se usa la carpeta actual de trabajo.
try:
    CARPETA_PROYECTO = Path(__file__).resolve().parent
except NameError:
    CARPETA_PROYECTO = Path.cwd()

CARPETA_ENTRADA = CARPETA_PROYECTO / "server_inputs"
CARPETA_SALIDA = CARPETA_PROYECTO / "server_outputs"

ARCHIVO_ENTRADA = CARPETA_ENTRADA / "file.ope"
ARCHIVO_CLIENTE_CSV = CARPETA_SALIDA / "cliente.csv"
ARCHIVO_DEUDA_CSV = CARPETA_SALIDA / "deuda.csv"
ARCHIVO_DEUDA_DB = CARPETA_SALIDA / "deuda.db"

# Columnas del DataFrame cliente (en el orden indicado por el enunciado)
COLUMNAS_CLIENTE = [
    "SBSCodigoCliente", "SBSFechaReporte", "SBSTipoDocumentoT",
    "SBSRucCliente", "SBSTipoDocumento", "SBSNumeroDocumento",
    "SBSTipoPer", "SBSTipoEmpresa", "SBSNumeroEntidad", "SBSSalNor",
    "SBSSalCPP", "SBSSalDEF", "SBSSalDUD", "SBSSalAPER", "SBSAPEPAT",
    "SBSAPEMAT", "SBSAPECAS", "SBSNOMCLI", "SBSNOMCLI2",
]

# Columnas finales del DataFrame deuda (después del renombrado)
COLUMNAS_DEUDA_FINAL = [
    "Cod_SBS", "Cod_Emp", "Tip_Credit", "Nivel2", "Moneda",
    "SubCodigoCuenta", "Condicion", "Val_Saldo", "Clasif_Deu", "Cod_Cuenta",
]

SEPARADOR = "=" * 50


def validar(condicion, descripcion):
    """Muestra [OK] si la condición se cumple; si no, detiene el programa."""
    if condicion:
        print(f"  [OK] {descripcion}")
    else:
        raise ValueError(f"VALIDACIÓN FALLIDA: {descripcion}")


print(SEPARADOR)
print("PROYECTO ETL - file.ope")
print(SEPARADOR)

PROYECTO ETL - file.ope


## EXTRACT
Leer los datos desde el servidor de entrada (`server_inputs`).

In [2]:
# ============================================================================
# EXTRACT: leer los datos desde el servidor de entrada (server_inputs)
# ============================================================================
print("\nEXTRACT")

# Validación 1: el archivo de entrada debe existir
validar(ARCHIVO_ENTRADA.exists(), f"Existe el archivo {ARCHIVO_ENTRADA.name}")

# El archivo está codificado en UTF-8 (tiene letras como Ñ, Ó, Ú)
with open(ARCHIVO_ENTRADA, encoding="utf-8") as archivo:
    todas_las_lineas = archivo.read().splitlines()

# Quitamos las líneas vacías
lineas_utiles = [linea for linea in todas_las_lineas if linea.strip() != ""]

# Separamos las líneas según su primer carácter:
#   "1" -> registro de cliente
#   "2" -> registro de deuda
#   otro -> se ignora (por ejemplo, la cabecera "Field_1")
registros_cliente = []
registros_deuda = []
lineas_ignoradas = []

for linea in lineas_utiles:
    if linea.startswith("1"):
        registros_cliente.append(linea)
    elif linea.startswith("2"):
        registros_deuda.append(linea)
    else:
        lineas_ignoradas.append(linea)

print(f"  Total de líneas útiles leídas: {len(lineas_utiles)}")
print(f"  Registros de cliente (empiezan en 1): {len(registros_cliente)}")
print(f"  Registros de deuda (empiezan en 2): {len(registros_deuda)}")
print(f"  Líneas ignoradas: {len(lineas_ignoradas)} {lineas_ignoradas}")

# Validaciones 2 y 3: se encontraron clientes y deudas
validar(len(registros_cliente) > 0, "Se encontraron registros de cliente")
validar(len(registros_deuda) > 0, "Se encontraron registros de deuda")


EXTRACT
  [OK] Existe el archivo file.ope
  Total de líneas útiles leídas: 1001
  Registros de cliente (empiezan en 1): 139
  Registros de deuda (empiezan en 2): 861
  Líneas ignoradas: 1 ['Field_1']
  [OK] Se encontraron registros de cliente
  [OK] Se encontraron registros de deuda


## TRANSFORM
Construir los DataFrames `cliente` y `deuda`.

In [3]:
# ============================================================================
# TRANSFORM: construir los DataFrames cliente y deuda
# ============================================================================
print("\nTRANSFORM")

# ---------------------------------------------------------------------------
# DataFrame cliente
# ---------------------------------------------------------------------------
# El primer carácter ("1") solo indica el tipo de registro, por eso se quita.
# Ejemplo: "10038518267|20200930|..." -> SBSCodigoCliente = "0038518267"
# Luego se separan los campos usando el carácter "|".
filas_cliente = []

for numero, registro in enumerate(registros_cliente, start=1):
    campos = registro[1:].split("|")

    # Validación 4: cada registro debe tener exactamente 19 campos
    if len(campos) != len(COLUMNAS_CLIENTE):
        raise ValueError(
            f"El cliente N° {numero} tiene {len(campos)} campos "
            f"(se esperaban {len(COLUMNAS_CLIENTE)}): {registro}"
        )
    filas_cliente.append(campos)

# Todos los valores se guardan como texto para no perder ceros iniciales
cliente = pd.DataFrame(filas_cliente, columns=COLUMNAS_CLIENTE)

# ---------------------------------------------------------------------------
# DataFrame deuda
# ---------------------------------------------------------------------------
# Los campos son de ancho fijo. Igual que en cliente, se quita el primer
# carácter ("2") que solo indica el tipo de registro. Así, el carácter 1
# del enunciado es el primer carácter del código SBS.
#
# Python cuenta desde 0 y el final del slice NO se incluye, por lo tanto:
#   "del caracter 1 al 10"  ->  datos[0:10]
#   "del caracter 11 al 15" ->  datos[10:15]   ... y así sucesivamente.
filas_deuda = []

for registro in registros_deuda:
    datos = registro[1:]
    fila = {
        "CodigoSBS": datos[0:10],           # caracteres 1 al 10
        "CodigoEmpresa": datos[10:15],      # caracteres 11 al 15
        "TipoCredito": datos[15:17],        # caracteres 16 al 17
        "Nivel2": datos[17:19],             # caracteres 18 al 19
        "Moneda": datos[19:20],             # caracter 20
        "SubCodigoCuenta": datos[20:31],    # caracteres 21 al 31
        "Condicion": datos[31:37],          # caracteres 32 al 37
        "ValorSaldo": datos[37:41],         # caracteres 38 al 41
        "ClasificacionDeuda": datos[41:42], # caracter 42
    }
    filas_deuda.append(fila)

deuda = pd.DataFrame(filas_deuda)

# CodigoCuenta = Nivel2 + Moneda + SubCodigoCuenta (unión de TEXTO, no suma)
deuda["CodigoCuenta"] = deuda["Nivel2"] + deuda["Moneda"] + deuda["SubCodigoCuenta"]

# Renombrado de columnas pedido por el enunciado
deuda = deuda.rename(columns={
    "CodigoSBS": "Cod_SBS",
    "CodigoEmpresa": "Cod_Emp",
    "TipoCredito": "Tip_Credit",
    "ValorSaldo": "Val_Saldo",
    "ClasificacionDeuda": "Clasif_Deu",
    "CodigoCuenta": "Cod_Cuenta",
})

print(f"  DataFrame cliente: {cliente.shape[0]} filas x {cliente.shape[1]} columnas")
print(f"  DataFrame deuda:   {deuda.shape[0]} filas x {deuda.shape[1]} columnas")

# Validación 4: columnas de cliente
validar(list(cliente.columns) == COLUMNAS_CLIENTE,
        "cliente tiene las 19 columnas en el orden correcto")
validar(len(cliente) == len(registros_cliente),
        "cliente tiene una fila por cada registro que empieza en 1")

# Validación 5: columnas de deuda
validar(list(deuda.columns) == COLUMNAS_DEUDA_FINAL,
        "deuda tiene las 10 columnas finales (con los 6 renombrados)")
validar(len(deuda) == len(registros_deuda),
        "deuda tiene una fila por cada registro que empieza en 2")

# Validación 6: no se pierden los ceros iniciales (los códigos miden 10)
validar(cliente["SBSCodigoCliente"].str.len().eq(10).all()
        and deuda["Cod_SBS"].str.len().eq(10).all(),
        "Los códigos SBS conservan sus 10 dígitos (ceros iniciales incluidos)")

# Validación 7: Cod_Cuenta está bien construido.
# Nivel2, Moneda y SubCodigoCuenta están juntos en el archivo (caracteres
# 18 al 31), así que Cod_Cuenta debe ser igual a ese tramo del registro.
tramo_original = [registro[1:][17:31] for registro in registros_deuda]
validar(list(deuda["Cod_Cuenta"]) == tramo_original,
        "Cod_Cuenta = Nivel2 + Moneda + SubCodigoCuenta (14 caracteres)")

# Observación: el enunciado define 42 caracteres, pero los registros reales
# de deuda son más largos. Esos caracteres extra no se usan (ver README).
longitud_maxima = max(len(registro[1:]) for registro in registros_deuda)
print(f"  Observación: los registros de deuda tienen hasta {longitud_maxima} caracteres "
      f"(sin el '2'); el enunciado solo define los primeros 42.")


TRANSFORM
  DataFrame cliente: 139 filas x 19 columnas
  DataFrame deuda:   861 filas x 10 columnas
  [OK] cliente tiene las 19 columnas en el orden correcto
  [OK] cliente tiene una fila por cada registro que empieza en 1
  [OK] deuda tiene las 10 columnas finales (con los 6 renombrados)
  [OK] deuda tiene una fila por cada registro que empieza en 2
  [OK] Los códigos SBS conservan sus 10 dígitos (ceros iniciales incluidos)
  [OK] Cod_Cuenta = Nivel2 + Moneda + SubCodigoCuenta (14 caracteres)
  Observación: los registros de deuda tienen hasta 54 caracteres (sin el '2'); el enunciado solo define los primeros 42.


## LOAD
Guardar los resultados en el servidor de salida (`server_outputs`).

In [4]:
# ============================================================================
# LOAD: guardar los DataFrames en el servidor de salida (server_outputs)
# ============================================================================
print("\nLOAD")

CARPETA_SALIDA.mkdir(exist_ok=True)

# Archivos CSV ("utf-8-sig" permite que Excel muestre bien las tildes y la Ñ)
cliente.to_csv(ARCHIVO_CLIENTE_CSV, index=False, encoding="utf-8-sig")
deuda.to_csv(ARCHIVO_DEUDA_CSV, index=False, encoding="utf-8-sig")

# Validación 8: los CSV existen y tienen el mismo contenido que los DataFrames.
# Se leen como texto (dtype=str) para comprobar que los ceros se conservan.
for archivo_csv, dataframe in [(ARCHIVO_CLIENTE_CSV, cliente),
                               (ARCHIVO_DEUDA_CSV, deuda)]:
    validar(archivo_csv.exists() and archivo_csv.stat().st_size > 0,
            f"{archivo_csv.name} creado y no está vacío")
    csv_leido = pd.read_csv(archivo_csv, dtype=str, keep_default_na=False,
                            encoding="utf-8-sig")
    validar(list(csv_leido.columns) == list(dataframe.columns)
            and len(csv_leido) == len(dataframe)
            and csv_leido.values.tolist() == dataframe.values.tolist(),
            f"{archivo_csv.name}: {len(csv_leido)} filas, columnas y datos correctos")

# BONO: guardar la deuda en una base de datos SQLite.
# if_exists="replace" evita duplicar registros si se ejecuta el ETL otra vez.
conexion = sqlite3.connect(ARCHIVO_DEUDA_DB)
deuda.to_sql("deuda", conexion, if_exists="replace", index=False)

cantidad_sqlite = conexion.execute("SELECT COUNT(*) FROM deuda").fetchone()[0]
primer_codigo_sqlite = conexion.execute("SELECT Cod_SBS FROM deuda LIMIT 1").fetchone()[0]
conexion.close()

# Validaciones 9 y 10: la base existe y tiene la misma cantidad de filas
validar(ARCHIVO_DEUDA_DB.exists(), f"{ARCHIVO_DEUDA_DB.name} creado con la tabla deuda")
validar(cantidad_sqlite == len(deuda),
        f"Registros en SQLite ({cantidad_sqlite}) = filas del DataFrame deuda ({len(deuda)})")
validar(primer_codigo_sqlite == deuda.loc[0, "Cod_SBS"],
        f"SQLite conserva los ceros iniciales (primer Cod_SBS = {primer_codigo_sqlite})")


print("\n" + SEPARADOR)
print("ETL FINALIZADO CORRECTAMENTE")
print(SEPARADOR)


LOAD
  [OK] cliente.csv creado y no está vacío
  [OK] cliente.csv: 139 filas, columnas y datos correctos
  [OK] deuda.csv creado y no está vacío
  [OK] deuda.csv: 861 filas, columnas y datos correctos


  [OK] deuda.db creado con la tabla deuda
  [OK] Registros en SQLite (861) = filas del DataFrame deuda (861)
  [OK] SQLite conserva los ceros iniciales (primer Cod_SBS = 0038518267)

ETL FINALIZADO CORRECTAMENTE


## Vista previa de los DataFrames

In [5]:
cliente.head(3)

,SBSCodigoCliente,SBSFechaReporte,SBSTipoDocumentoT,SBSRucCliente,SBSTipoDocumento,SBSNumeroDocumento,SBSTipoPer,SBSTipoEmpresa,SBSNumeroEntidad,SBSSalNor,SBSSalCPP,SBSSalDEF,SBSSalDUD,SBSSalAPER,SBSAPEPAT,SBSAPEMAT,SBSAPECAS,SBSNOMCLI,SBSNOMCLI2
0,0038518267,20200930,,,1,29409477,1,,002,00000,00000,00000,00000,10000,,BAUTISTA,,JUANA,
1,0029280002,20200930,,,1,06146421,1,,001,00000,00000,00000,00000,10000,,BUSTAMANTE,,JOSE,LUIS
2,0016584398,20200930,,,1,07699936,1,,001,00000,00000,00000,00000,10000,,CASTILLO,,ARMANDO,NOLBERTO


In [6]:
deuda.head(3)

,Cod_SBS,Cod_Emp,Tip_Credit,Nivel2,Moneda,SubCodigoCuenta,Condicion,Val_Saldo,Clasif_Deu,Cod_Cuenta
0,0038518267,00145,11,14,1,50302020000,014900,0000,0,14150302020000
1,0038518267,00140,11,81,1,30200000000,042700,0000,0,81130200000000
2,0038518267,00145,11,14,1,90301000000,000000,0000,0,14190301000000


## Consulta a la base de datos SQLite (bono)

Se leen las 3 primeras filas de la tabla `deuda` guardada en `server_outputs/deuda.db`.

In [7]:
conexion = sqlite3.connect(ARCHIVO_DEUDA_DB)
consulta_sqlite = pd.read_sql("SELECT * FROM deuda LIMIT 3", conexion)
conexion.close()
consulta_sqlite

,Cod_SBS,Cod_Emp,Tip_Credit,Nivel2,Moneda,SubCodigoCuenta,Condicion,Val_Saldo,Clasif_Deu,Cod_Cuenta
0,0038518267,00145,11,14,1,50302020000,014900,0000,0,14150302020000
1,0038518267,00140,11,81,1,30200000000,042700,0000,0,81130200000000
2,0038518267,00145,11,14,1,90301000000,000000,0000,0,14190301000000


## Descargar los resultados (opcional, solo Google Colab)

Los archivos de Colab se borran al cerrar la sesión. Esta celda descarga `cliente.csv`, `deuda.csv` y `deuda.db` a tu computadora.

In [ ]:
try:
    from google.colab import files
    for nombre in ["cliente.csv", "deuda.csv", "deuda.db"]:
        files.download(f"server_outputs/{nombre}")
except ImportError:
    print("No estás en Google Colab: los archivos ya están en la carpeta server_outputs.")